# CONTACT MAPS CALCULATION AND ANALYSIS

In [1]:
import pandas as pd
import structure_validation
import structural_analysis
import contact_maps
import importlib
import scipy.stats as stats

In [2]:
df_allsp = pd.read_csv("files/anks_final_aln.sto_prointvar_structure_table_with_validation.csv") # STRUCTURAL DATA TABLE
aln_in = "files/anks_final_aln.sto" # MSA
aln_fmt = "stockholm" # MSA FORMAT

2025-11-12 13:27:59,284 - WARNING - /tmp/ipykernel_6137/77797086.py:1: DtypeWarning: Columns (2,15,16,19,20,26,27,29,30,32,34,42,43,44,45,46,47,48,49,52,53,64,65,66,68,70,72,76,77,78,92,96,114,116,119,120,123,124,129,130,133,134,143,144,150,152,154,156,158,162,164,170,207,209,214,215,231,232,233,234,235,266,277,278,281,287,288,289,292,303,306,315) have mixed types. Specify dtype option on import or set low_memory=False.
  df_allsp = pd.read_csv("files/anks_final_aln.sto_prointvar_structure_table_with_validation.csv") # STRUCTURAL DATA TABLE
 


In [3]:
df_allsp_rsrz_filt = structure_validation.filter_rsrz_rscc(df_allsp) # FILTERS STRUCTURAL TABLE SO WE KEEP HIGH QUALITY RESIDUES MEETING OUR RSCC AND RSRZ THRESHOLDS
cons_cols_allsp = structural_analysis.get_cons_cols(aln_in, aln_fmt) # GETS CONSENSUS COLUMNS FROM MSA (OCCUPANCY > 0.5)
df_rf = contact_maps.format_df(df_allsp_rsrz_filt, cons_cols_allsp) # FORMATS STRUCTURAL TABLE FOR FOLLOW-UP ANALYSIS

cons_cols_eq = contact_maps.get_cons_cols_eq(cons_cols_allsp) # EQUIVALENCE BETWEEN CONSENSUS ALIGNMENT COLUMN NUMBERS AND DOMAIN POSITIONS (1-33)
df_cols_eq = contact_maps.get_df_cols_eq(df_rf)

2025-11-12 13:28:00,720 - WARNING - /workspaces/Student-Pfam-Workflow/contact_maps.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2_filt.Alignment_column_A = df2_filt.Alignment_column_A.astype(int)
 
2025-11-12 13:28:00,726 - WARNING - /workspaces/Student-Pfam-Workflow/contact_maps.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2_filt.Alignment_column_B = df2_filt.Alignment_column_B.astype(int)
 
2025-11-12 13:28:00,728 - WARNING - /workspaces/Student-Pfam-Workflow/contact_maps.py:22: Settin

## INTRA-ANK REPEAT INTERACTIONS

In [4]:
intra_res_occ = contact_maps.get_intra_cons_occ(df_rf, cons_cols_eq) # GETS COVERAGE OF EACH INTRA-REPEAT POSITION PAIR IN OUR STRUCTURAL DATASET

The dataframe contains information of:
26668 residues, 10571 of which are unique
173 PDB structures
72 different proteins
382 unique repeats


In [5]:
intra = contact_maps.get_intra_cons(df_rf, cons_cols_eq, t = 0) # CALCULATES ABSOLUTE FREQUENCY OF INTRA-REPEAT CONTACTS
intra_norm = contact_maps.normalize_contacts(intra, intra_res_occ) # NORMALISES CONTACT MAP BY POSITION PAIR COVERAGE IN DATASET

In [6]:
contact_maps.plot_intra_cons(intra_norm, "viridis")

### INTRA-ANK CONTACTS BETWEEN RESIDUES FURTHER APPART THAN 5 RESIDUES

In [7]:
intra_t5 = contact_maps.get_intra_cons(df_rf, cons_cols_eq, t = 5) # ONLY PAIRS OF POSITIONS FURTHER AWAY THAN 5 POSITIONS FROM EACH OTHER
intra_norm_t5 = contact_maps.normalize_contacts(intra_t5, intra_res_occ) # NORMALISATION

In [8]:
contact_maps.plot_intra_cons(intra_norm_t5, "viridis")

In [9]:
occ_intra = contact_maps.get_tot_occ_res(intra_res_occ, len(intra), contact_map = "intra", t = 5) # STRUCTURAL COVERAGE PER POSITION
cons_intra = contact_maps.get_tot_cons_res(intra, contact_map = "intra", t = 5) # TOTAL NUMBER OF CONTACTS PER POSITION
enrichment_df_intra = contact_maps.get_OR_from_cons(cons_intra, occ_intra) # CALCULATES ENRICHMENT SCORES IN INTRA-REPEAT CONTACTS, P-VALUE, AND 95% CI FOR ENRICHMENT SCORE
enrichment_df_intra = structural_analysis.add_miss_class(enrichment_df_intra) # ADDS MISSENSE ENRICHMENT CLASS

### ENRICHMENT IN INTRA-REPEAT CONTACTS

In [10]:
cmd_cons = enrichment_df_intra[enrichment_df_intra.color_class == "royalblue"].contacts.sum()
cmd_occ = enrichment_df_intra[enrichment_df_intra.color_class == "royalblue"].occ.sum()
rest_cons = enrichment_df_intra[enrichment_df_intra.color_class != "royalblue"].contacts.sum()
rest_occ = enrichment_df_intra[enrichment_df_intra.color_class != "royalblue"].occ.sum()

In [11]:
oddsr, pval = stats.fisher_exact([[cmd_cons, rest_cons], [cmd_occ, rest_occ]]) # CMDs ARE ENRICHED IN INTRA-REPEAT CONTACTS RELATIVE TO REST OF ANK POSITIONS
oddsr, pval

(2.7785925828932565, 0.0)

In [12]:
contact_maps.plot_cons_enrichment(enrichment_df_intra, class_col = "class", color_col = "color_class")

## INTER-ANK REPEAT INTERACTIONS

In [13]:
inter_res_occ = contact_maps.get_inter_cons_occ(df_rf, cons_cols_eq)
inter = contact_maps.get_inter_cons(df_rf, cons_cols_eq, df_cols_eq)
inter_norm = contact_maps.normalize_contacts(inter, inter_res_occ)

The dataframe contains information of:
26668 residues, 10571 of which are unique
173 PDB structures
72 different proteins
382 unique repeats
310 pairs of ARs were used


In [14]:
contact_maps.plot_inter_cons(inter_norm, "viridis")

### ENRICHMENT IN INTER-REPEAT CONTACTS

In [15]:
occ_inter = contact_maps.get_tot_occ_res(inter_res_occ, len(inter), contact_map = "inter", t = 0) # CALCULATES THE ABSOLUTE STRUCTURAL COVERAGE PER DOMAIN POSITION
cons_inter = contact_maps.get_tot_cons_res(inter, contact_map = "inter", t = 0) # CALCULATES THE ABSOLUTE NUMBER OF CONTACTS PER DOMAIN POSITION
enrichment_df_inter = contact_maps.get_OR_from_cons(cons_inter, occ_inter) # CALCULATES OR, LOG(OR), P-VALUE AND 95% CI FOR THE ENRICHMENT IN CONTACTS
enrichment_df_inter = structural_analysis.add_miss_class(enrichment_df_inter) # ADDS MISSENSE ENRICHMENT CLASS

In [16]:
enrichment_df_inter.head()

,occ,contacts,oddsratio,log_oddsratio,pvalue,ci_dist,class,color_class
1,16281,379.0,0.456804,-0.783501,1.748112e-64,0.102569,UMD,firebrick
2,17224,528.0,0.603793,-0.504523,3.398669e-34,0.087453,None,grey
3,17445,1065.0,1.226282,0.203987,6.784342e-10,0.063082,UMD,firebrick
4,17855,1250.0,1.415034,0.347154,1.487260e-28,0.058664,CME,green
5,17976,1678.0,1.917481,0.651012,1.142598e-115,0.051561,CME,green


In [17]:
cmd_inter_cons = enrichment_df_inter[enrichment_df_inter.color_class == "royalblue"].contacts.sum()
cmd_inter_occ = enrichment_df_inter[enrichment_df_inter.color_class == "royalblue"].occ.sum()
rest_inter_cons = enrichment_df_inter.contacts.sum() - cmd_inter_cons
rest_inter_occ = enrichment_df_inter.occ.sum() - cmd_inter_occ

In [18]:
oddsr, pval = stats.fisher_exact([[cmd_inter_cons, rest_inter_cons], [cmd_inter_occ, rest_inter_occ]]) # CMDs ARE NOT ENRICHED IN INTER-REPEAT CONTACTS
oddsr, pval

(0.8532348751752032, 7.464855076360734e-20)

In [19]:
contact_maps.plot_cons_enrichment(enrichment_df_inter, contact_map = "Inter", class_col = "class", color_col = "color_class")